# Arabic OCR — prepare & vet training data (CPU)

No GPU. Assemble the sources named in a `configs/training/*.yaml`, run the
contamination check against the benchmark test splits, and eyeball a sample.
Use this before spending GPU time in `05_colab_train_lora.ipynb`.

## 1. Repo

In [ ]:
# Bulletproof checkout: works whether the repo is missing, at the wrong
# path, or has local edits. Always ends up at the latest origin/main.
import os
REPO = '/content/arabic-ocr-benchmark'
token = None
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception as e:
    print('no GH_TOKEN secret:', e)
url = f"https://{(token + '@') if token else ''}github.com/rabeeh97/arabic-ocr-benchmark.git"
if os.path.isdir(REPO + '/.git'):
    os.system(f'git -C {REPO} fetch --depth 1 origin main && git -C {REPO} reset --hard origin/main')
    # drop untracked leftovers (e.g. a stale sweep_manifest.json) so the
    # manifest is always exactly what's in the repo. Keeps gitignored _cache.
    os.system(f'git -C {REPO} clean -fd -- results')
else:
    os.system(f'rm -rf {REPO} && git clone --depth 1 {url} {REPO}')
os.chdir(REPO)
# src-layout package: importable straight from the checkout even if the
# `pip install -e` cell fails or is skipped (this is what caused the old
# 'No module named arabic_ocr_bench' in the sweep cell).
import sys
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
print(os.popen('git log --oneline -3').read())

## 2. Install — core + train extras (no torch) + Arabic fonts

In [ ]:
# Keep all downloads on Colab's own scratch disk (ephemeral, ~78 GB) --
# never the local machine. Nothing here touches your computer.
import os
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Optional HF_TOKEN secret: lifts anonymous rate limits and speeds the
# Muharaf download (and is required for gated models like PaliGemma's base).
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
    if _hf:
        os.environ['HF_TOKEN'] = _hf
        print('HF_TOKEN set')
except Exception:
    pass

In [ ]:
!pip install -q -e '.[train]'   # datasets, imagehash, fonttools, pydantic — no torch
!python scripts/fetch_fonts.py   # extra Arabic faces for synthetic rendering (Amiri family)

## 3. Config

In [ ]:
TRAIN_CFG = 'configs/training/m1_printed_synth_qwen25vl3b.yaml'
from arabic_ocr_bench.training.config import load_training_config
cfg = load_training_config(TRAIN_CFG)
print(cfg.experiment, [s.name for s in cfg.data.sources])

## 4. (synthetic sources) mount Drive + set up a resumable render cache

For a `synthetic_printed` source every line is rendered once, then reused by
`05_colab_train_lora.ipynb`. Rendering ~15k lines is slow, so this cell makes it
**resumable and Drive-friendly**:

* `work_dir` &rarr; `/content` (fast local disk) is where images are actually written.
* `cache_dir` &rarr; a Drive folder holds a single `images.tar` checkpoint plus a
  cached `pool_*.json` (the corpus + contamination step, otherwise re-run every
  session).

On start it restores `images.tar` from Drive; it re-checkpoints every
`flush_every` newly-rendered lines and once at the end. A disconnect costs at
most `flush_every` lines &mdash; just re-run cell&nbsp;5 and it picks up where it
stopped. One big sequential Drive read/write instead of 15k tiny-file
round-trips. Skip this cell if the config has no synthetic source.

In [ ]:
import pathlib
SYNTH = [s for s in cfg.data.sources if 'synth' in s.adapter.lower()]
if SYNTH:
    from google.colab import drive; drive.mount('/content/drive')
    drive_cache = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench/synth') / cfg.experiment
    work = pathlib.Path('/content/synth') / cfg.experiment          # fast local scratch
    drive_cache.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)
    for s in SYNTH:
        s.params['cache_dir'] = str(drive_cache)   # durable: images.tar + pool cache
        s.params['work_dir'] = str(work)           # where rendering actually happens
        s.params['flush_every'] = 1000             # checkpoint to Drive every N new lines
    print('synth render ->', work, '| checkpoint ->', drive_cache / 'images.tar')


## 5. Assemble + class balance by source

In [ ]:
from arabic_ocr_bench.training.data import assemble_dataset
# Slow the first time (renders ~n synthetic lines, progress bar below). Safe to
# re-run after a disconnect: cached lines are skipped, only the tail is rendered.
bundle = assemble_dataset(cfg.data, seed=cfg.train.seed)
from collections import Counter
print('train', len(bundle['train']), 'val', len(bundle['val']))
print(Counter(r['source'] for r in bundle['train'] + bundle['val']))


## 6. Contamination report

In [ ]:
from arabic_ocr_bench.training.contamination import check_datasets
specs = [{**s.model_dump(exclude_none=True), 'name': s.name} for s in cfg.data.sources]
rep = check_datasets(specs, cfg.eval_datasets, limit=None)
import json; print(json.dumps(rep['pairs'], ensure_ascii=False, indent=2)[:4000])
print('flagged pairs:', rep['n_flagged'])

## 7. Eyeball samples

In [ ]:
import random
for r in random.Random(0).sample(bundle['train'], 8):
    display(r['images'][0])
    print(r['source'], '|', r['messages'][2]['content'][0]['text'])

## 8. Commit the contamination report

It's cited in every experiment log and the writeup — keep it in git.

In [ ]:
import subprocess
if token:
    subprocess.run(['git','add','-f','results/contamination_report.json'], check=True)
    r = subprocess.run(['git','commit','-m',f'contam: {cfg.experiment}'], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    print(subprocess.run(['git','push','origin','HEAD:main'], capture_output=True, text=True).stdout)